# Solutions to Exercises in Basic Probability

Given the joint probability distribution `JointPDF[x][y][z]` of a binary variable $X$,  a three-valued variable $Y$, and a binary variable $Z$ perform the following computations:

[Recall that numpy arrays can be summed using `numpy.array.sum`]

In [1]:
import numpy as np
JointPDF = np.array(
    [
        [
            [0.1, 0.05],
            [0.11, 0.12],
            [0.07,0.03]
        ],
        [
            [0.06, 0.16],
            [0.01, 0.08],
            [0.19,0.02]
        ]
     ]
     )

1. Verify that `JointPDF` is normalised.

In [2]:
JointPDF.sum(axis=(0,1,2))

np.float64(1.0)

2. Compute $P(X)$, $P(Y)$ and $P(Z)$

In [3]:
# Sum over the other axes
PX = JointPDF.sum(axis=(1,2))
PY = JointPDF.sum(axis=(0,2))
PZ = JointPDF.sum(axis=(0,1))

print(PX)
print(PY)
print(PZ)

[0.48 0.52]
[0.37 0.32 0.31]
[0.54 0.46]


3. Compute $P(X,Y)$, $P(Y,Z)$, $P(X,Z)$

In [4]:
PXY = JointPDF.sum(axis=2)
PXZ = JointPDF.sum(axis=1)
PYZ = JointPDF.sum(axis=0)
print(PXY)
print(PXZ)
print(PYZ)

[[0.15 0.23 0.1 ]
 [0.22 0.09 0.21]]
[[0.28 0.2 ]
 [0.26 0.26]]
[[0.16 0.21]
 [0.12 0.2 ]
 [0.26 0.05]]


4. Compute $P(X\vert Y)$

In [5]:

PX_y0 = PXY[:,0]/PXY[:,0].sum()
PX_y1 = PXY[:,1]/PXY[:,1].sum()
PX_y2 = PXY[:,2]/PXY[:,2].sum()


print(PX_y0)
print(PX_y1)
print(PX_y2)

[0.40540541 0.59459459]
[0.71875 0.28125]
[0.32258065 0.67741935]


5. **Bayesian reasoning: which factory?** A company makes components in two factories. Factory A makes 60% of all components and Factory B makes the remaining 40%. 2% of the components made by A are defective, whereas 5% of the components made by B are defective. A component is chosen at random from the company's total output.

   a) Express the *prior* $P(F)$ and the *likelihood* $P(\mathrm{Defective}\vert F)$ as numpy arrays.

   b) Compute the probability that a randomly chosen component is defective, $P(\mathrm{Defective})$.

   c) The chosen component is found to be defective. Use Bayes rule to compute the posterior $P(F\vert\mathrm{Defective})$. Which factory is the more likely source?

   d) A second component is taken from the *same* (unknown) factory and is also found to be defective. Use your posterior from (c) as the prior to compute the new posterior. Check your answer by computing the posterior in a single step from the original prior and the likelihood of *both* observations. [Hint: the two observations are conditionally independent given the factory.]

In [6]:
# a) Prior and likelihood (index 0: Factory A, index 1: Factory B)
PF = np.array([0.6, 0.4])
PD_F = np.array([0.02, 0.05])   # P(Defective | F)
print(PF, PD_F)

[0.6 0.4] [0.02 0.05]


b) By marginalisation, $P(\mathrm{Defective})=\sum_F P(\mathrm{Defective}\vert F)P(F)$

In [7]:
PD = (PD_F*PF).sum()
print(PD)

0.032


c) By Bayes rule, $P(F\vert \mathrm{Defective}) = P(\mathrm{Defective}\vert F)P(F)/P(\mathrm{Defective})$. Although Factory A makes more of the components, Factory B has a higher defect rate and is the more likely source (62.5%).

In [8]:
PF_D = PD_F*PF/PD
print(PF_D)

[0.375 0.625]


d) Sequential update: use the posterior from (c) as the new prior.

In [9]:
PD2 = (PD_F*PF_D).sum()
PF_D1D2 = PD_F*PF_D/PD2
print("Sequential update:", PF_D1D2)

# Single step: P(F|D1,D2) is proportional to P(D1|F)P(D2|F)P(F)
unnorm = PD_F*PD_F*PF
print("Single step:      ", unnorm/unnorm.sum())

Sequential update: [0.19354839 0.80645161]
Single step:       [0.19354839 0.80645161]


6. **Bayesian reasoning: the Monty Hall problem.** In a game show there are three doors. A car is hidden behind one of them (all equally likely) and goats are behind the other two. You choose Door 1. The host, who knows where the car is, then opens one of the other doors to reveal a goat. If both of the other doors hide goats, the host chooses between them at random. The host opens Door 3.

   a) Let $C\in\{1,2,3\}$ be the door hiding the car and $H$ be the door opened by the host. Write down the prior $P(C)$ and the likelihood $P(H=3\vert C)$ for each value of $C$. [Think carefully about what the host is able to do in each case.]

   b) Compute the posterior $P(C\vert H=3)$ using Bayes rule. Should you stick with Door 1 or switch to Door 2?

   c) Verify your answer by simulating the game a large number of times using direct sampling (see `numpy.random`). Keep only those games in which you chose Door 1 and the host opened Door 3, and compute the fraction of these in which the car was behind each door.

a) The prior is uniform, $P(C)=(1/3,1/3,1/3)$. The host will never open Door 1 (our choice) or the door with the car. If $C=1$, both Doors 2 and 3 hide goats and the host chooses at random, so $P(H=3\vert C=1)=1/2$. If $C=2$, the host *must* open Door 3, so $P(H=3\vert C=2)=1$. If $C=3$ the host cannot open Door 3, so $P(H=3\vert C=3)=0$.

In [10]:
# PC[c]: prior probability that the car is behind door c+1
PC = np.ones(3)/3

# PH_C[h][c] = P(H | C): rows are h = host opens door 2, 3; columns are c = car behind door 1, 2, 3
PH_C = np.array([[0.5, 0.0, 1.0],
                 [0.5, 1.0, 0.0]])
print(PC)
print(PH_C)

[0.33333333 0.33333333 0.33333333]
[[0.5 0.  1. ]
 [0.5 1.  0. ]]


b) Apply Bayes rule, where the denominator is the sum of the numerators: $P(H=3)=\sum_C P(H=3\vert C)P(C)=1/2$.

In [11]:
PH = PH_C @ PC            # P(H), by marginalising over C
PC_H = (PH_C*PC).T/PH     # P(C | H) by Bayes rule, indexed PC_H[c][h]
print(PH)
print(PC_H[:,1])          # P(C | H=3)

[0.5 0.5]
[0.33333333 0.66666667 0.        ]


The posterior is $(1/3, 2/3, 0)$. You should **switch**: Door 2 hides the car with probability 2/3. The intuitive answer of 1/2 each is wrong because the host's choice is not random: it was constrained by the location of the car, so it carries information. In the case $C=2$ the host had no choice, whereas in the case $C=1$ he had a choice and so opened Door 3 only half of the time.

c) Simulation. We sample the game directly from the generative model $P(C)P(H\vert C)$, discard all games that are not consistent with the evidence (rejection sampling) and count.

In [12]:
rng = np.random.default_rng(42)
N = 200000
counts = np.zeros(3)
nkept = 0
for _ in range(N):
    car = rng.integers(1, 4)               # car behind door 1, 2 or 3
    # Host opens a door that is neither the player's choice (1) nor the car
    options = [d for d in (2, 3) if d != car]
    host = rng.choice(options)
    if host == 3:                          # keep only games consistent with the evidence
        counts[car-1] += 1
        nkept += 1
print(counts/nkept)

[0.33297059 0.66702941 0.        ]
